# 01 — Data loading (GSE40279)

Downloads GEO metadata and the supplementary beta matrix, aligns samples with ages, and exports processed **`X`**, **`y`**, and **`metadata`** for downstream notebooks.

**Does not** train any model.

## Paths and folders

All paths are relative to the project root (`Path("..")` from this notebook).

In [1]:
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

for path in [DATA_DIR, RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_DIR: C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction
PROCESSED_DIR: C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\processed


## Dependencies

In [2]:
%pip install -q GEOparse pyarrow

Note: you may need to restart the kernel to use updated packages.


## Load GEO series (brief SOFT)

In [3]:
import GEOparse

gse = GEOparse.get_GEO(geo="GSE40279", destdir=str(RAW_DIR), how="brief")
print(gse)
print("Samples:", len(gse.gsms))

10-May-2026 15:45:21 DEBUG utils - Directory C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\raw already exists. Skipping.
10-May-2026 15:45:21 INFO GEOparse - File already exist: using local version.
10-May-2026 15:45:21 INFO GEOparse - Parsing C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\raw\GSE40279.txt: 
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989827
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989828
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989829
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989830
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989831
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989832
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989833
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989834
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989835
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989836
10-May-2026 15:45:21 DEBUG GEOparse - SAMPLE: GSM989837
10-May-2026 15:45

<SERIES: None - 656 SAMPLES, 0 d(s)>
Samples: 656


## Parse sample metadata

In [4]:
import pandas as pd

rows = []
for gsm_id, gsm in gse.gsms.items():
    row = {"gsm_id": gsm_id}
    for item in gsm.metadata.get("characteristics_ch1", []):
        if ": " in item:
            key, value = item.split(": ", 1)
            row[key.strip().lower()] = value.strip()
    row["title"] = gsm.metadata.get("title", [None])[0]
    rows.append(row)

meta = pd.DataFrame(rows)
print(meta.shape)
display(meta.head())

(656, 8)


,gsm_id,age (y),source,plate,gender,ethnicity,tissue,title
0,GSM989827,67,UCSD,1,F,Caucasian - European,whole blood,age 67y 1001
1,GSM989828,89,UCSD,1,F,Caucasian - European,whole blood,age 89y 1002
2,GSM989829,66,UCSD,1,F,Caucasian - European,whole blood,age 66y 1003
3,GSM989830,64,UCSD,1,F,Caucasian - European,whole blood,age 64y 1004
4,GSM989831,62,UCSD,1,F,Caucasian - European,whole blood,age 62y 1005


## Clean metadata (numeric age, drop missing age)

In [5]:
meta_clean = meta.copy()
meta_clean["age"] = pd.to_numeric(meta_clean["age (y)"], errors="coerce")
meta_clean["plate"] = pd.to_numeric(meta_clean["plate"], errors="coerce")
meta_clean = meta_clean.dropna(subset=["age"]).reset_index(drop=True)
print(meta_clean.shape)
display(meta_clean.head())

(656, 9)


,gsm_id,age (y),source,plate,gender,ethnicity,tissue,title,age
0,GSM989827,67,UCSD,1,F,Caucasian - European,whole blood,age 67y 1001,67
1,GSM989828,89,UCSD,1,F,Caucasian - European,whole blood,age 89y 1002,89
2,GSM989829,66,UCSD,1,F,Caucasian - European,whole blood,age 66y 1003,66
3,GSM989830,64,UCSD,1,F,Caucasian - European,whole blood,age 64y 1004,64
4,GSM989831,62,UCSD,1,F,Caucasian - European,whole blood,age 62y 1005,62


## Download and load beta matrix (samples × CpGs)

In [6]:
import urllib.request

beta_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE40nnn/GSE40279/suppl/"
    "GSE40279_average_beta.txt.gz"
)
beta_path = RAW_DIR / "GSE40279_average_beta.txt.gz"
if not beta_path.exists():
    print("Downloading beta matrix...")
    urllib.request.urlretrieve(beta_url, beta_path)
else:
    print("Using existing beta file")
print("Size MB:", round(beta_path.stat().st_size / 1024 / 1024, 2))

Using existing beta file
Size MB: 1166.59


In [7]:
beta = pd.read_csv(beta_path, sep="\t", compression="gzip")
cpg_ids = beta["ID_REF"]
beta_t = beta.drop(columns=["ID_REF"]).T
beta_t.columns = cpg_ids
print("Beta orientation (samples × CpGs):", beta_t.shape)

Beta orientation (samples × CpGs): (656, 473034)


## Align metadata with beta rows and build `X`, `y`

Sample IDs in the beta file use codes like `X1001`; GEO titles end with the same numeric code.

In [8]:
meta_model = meta_clean.copy()
meta_model["sample_code"] = meta_model["title"].str.extract(r"(\d+)$")[0]
meta_model["sample_id_beta"] = "X" + meta_model["sample_code"]

common = sorted(set(meta_model["sample_id_beta"]) & set(beta_t.index))
beta_model = beta_t.loc[common].copy()
meta_aligned = meta_model.set_index("sample_id_beta").loc[common]
y = meta_aligned["age"].astype(float)
X = beta_model.astype("float32")

assert X.index.equals(y.index)
print("X:", X.shape, "y:", y.shape)

X: (656, 473034) y: (656,)


## Export processed files

- **`X.parquet`** — float matrix, index = sample IDs  
- **`y.csv`** — chronological age, same row order as `X`  
- **`metadata.csv`** — aligned clinical/metadata columns for each sample  
- **`n_cpgs.txt`** — number of CpG columns (used by notebook 03 for Fisher exact background without reloading `X`)


In [9]:
x_path = PROCESSED_DIR / "X.pkl.gz"
y_path = PROCESSED_DIR / "y.csv"
meta_path = PROCESSED_DIR / "metadata.csv"
n_cpgs_path = PROCESSED_DIR / "n_cpgs.txt"

X.to_pickle(x_path, compression="gzip")
y.to_frame("age").to_csv(y_path, index_label="sample_id_beta")
meta_aligned.reset_index().to_csv(meta_path, index=False)
n_cpgs_path.write_text(str(X.shape[1]), encoding="utf-8")

exports = [x_path, y_path, meta_path, n_cpgs_path]
print("Exported:")
for p in exports:
    print(" -", p.resolve())

Exported:
 - C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\processed\X.pkl.gz
 - C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\processed\y.csv
 - C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\processed\metadata.csv
 - C:\Users\Eva\Desktop\Longevity\Projects\epigenetic-clock-reproduction\data\processed\n_cpgs.txt
